In [1]:
import tensorflow as tf
import os
import glob
import numpy as np
from model import build_1d_cnn_model
from common import INPUT_SHAPE,OUTPUT_DIM_NOTES, fast_gpu_map,parse_example
cnn_model=build_1d_cnn_model(None,INPUT_SHAPE,OUTPUT_DIM_NOTES,False)
cnn_model.summary()
tf.keras.utils.plot_model(cnn_model,to_file='cnn_model.png',show_shapes=True)
# cnn_model.load_weights('/home/gerald/workspace/src/GuitarMidi-LV2/python/neuralnetmodelling/checkpoints/guitarmidi_epoch02_valAcc0.9551.weights.h5')#
cnn_model.load_weights('guitarmidi.weights.h5')

input_data_dir = '/home/gerald/workspace/src/GuitarMidi-LV2/python/neuralnetmodelling/data_slices'
input_filepaths = '/home/gerald/workspace/src/GuitarMidi-LV2/python/neuralnetmodelling/data_slices/training_subset/filtered_poly_data.tfrecord'#sorted(glob.glob(os.path.join(input_data_dir, '**', 'input', 'data.tfrecord'), recursive=True))
# train_dataset = tf.data.Dataset.from_tensor_slices((input_filepaths))
# train_dataset=train_dataset.shuffle(buffer_size=len(input_filepaths))
# train_dataset=train_dataset.take(100)
# train_dataset = train_dataset.map(tf_load_sample_from_files, num_parallel_calls=tf.data.AUTOTUNE)
def representative_data_gen():
    # Use TFRecordDataset to actually read the files
    # We only need a few samples to calibrate quantization
    raw_dataset = tf.data.TFRecordDataset(input_filepaths).map(parse_example).take(100)
    
    # Map using your existing loading function
    calib_dataset = raw_dataset.map(lambda path: fast_gpu_map(path, training=False))
    
    for input_value, _ in calib_dataset.batch(1):
        # input_value is the (312, 256, 1) tensor
        yield [input_value]

converter = tf.lite.TFLiteConverter.from_keras_model(cnn_model)

converter.optimizations = [ tf.lite.Optimize.DEFAULT]
# converter.representative_dataset = representative_data_gen
# converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
# converter.target_spec.supported_types = [tf.int8] 

# converter.target_spec.supported_types = [tf.float16]
# converter.target_spec._experimental_supported_accumulation_type = tf.dtypes.float16
# Perform the conversion
tflite_model = converter.convert()


# converter=tf.lite.TFLiteConverter.from_keras_model(cnn_model)
# converter.optimizations = [tf.lite.Optimize.DEFAULT]
# tflite_model=converter.convert()

with open('guitarmidi.tflite','wb') as f:
    f.write(tflite_model)
print("TFLite model saved as guitarmidi.tflite")

W0000 00:00:1772220258.874458   91365 gpu_device.cc:2456] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0a. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.
W0000 00:00:1772220258.903947   91365 gpu_device.cc:2456] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0a. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.
I0000 00:00:1772220258.998234   91365 gpu_device.cc:2040] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 1169 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 5080, pci bus id: 0000:01:00.0, compute capability: 12.0a


Initial input shape: (None, 296, 512)
After first Conv2D: (None, 148, 64)
Extracting string from filters 0 to 20
String 0 section shape: (None, 20, 64)
String 0 after first Conv1D: (None, 20, 128)
Extracting string from filters 21 to 40
String 21 section shape: (None, 19, 64)
String 21 after first Conv1D: (None, 19, 128)
Extracting string from filters 41 to 60
String 41 section shape: (None, 19, 64)
String 41 after first Conv1D: (None, 19, 128)
Extracting string from filters 61 to 76
String 61 section shape: (None, 15, 64)
String 61 after first Conv1D: (None, 15, 128)
Extracting string from filters 77 to 96
String 77 section shape: (None, 19, 64)
String 77 after first Conv1D: (None, 19, 128)
Extracting string from filters 97 to 147
String 97 section shape: (None, 50, 64)
String 97 after first Conv1D: (None, 50, 128)


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 296, 256,  │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape (Reshape)   │ (None, 296, 256,  │          0 │ input_layer[0][0] │
│                     │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 296, 32,   │        272 │ reshape[0][0]     │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ leaky_re_lu         │ (None, 296, 32,   │          0 │ conv2d[0][0]      │
│ (LeakyReLU)         │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_1 (Reshape) │ (None, 296, 512)  │          0 │ leaky_re_lu[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 296, 32)   │    114,720 │ reshape_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 296, 32)   │        128 │ conv1d[0][0]      │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ leaky_re_lu_1       │ (None, 296, 32)   │          0 │ batch_normalizat… │
│ (LeakyReLU)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ spatial_dropout1d   │ (None, 296, 32)   │          0 │ leaky_re_lu_1[0]… │
│ (SpatialDropout1D)  │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_1 (Conv1D)   │ (None, 296, 64)   │     14,400 │ spatial_dropout1… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 296, 64)   │        256 │ conv1d_1[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ leaky_re_lu_2       │ (None, 296, 64)   │          0 │ batch_normalizat… │
│ (LeakyReLU)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d       │ (None, 148, 64)   │          0 │ leaky_re_lu_2[0]… │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ spatial_dropout1d_1 │ (None, 148, 64)   │          0 │ max_pooling1d[0]… │
│ (SpatialDropout1D)  │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda (Lambda)     │ (None, 20, 64)    │          0 │ spatial_dropout1… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_1 (Lambda)   │ (None, 19, 64)    │          0 │ spatial_dropout1… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_2 (Lambda)   │ (None, 19, 64)    │          0 │ spatial_dropout1… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_3 (Lambda)   │ (None, 15, 64)    │          0 │ spatial_dropout1… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_4 (Lambda)   │ (None, 19, 64)    │          0 │ spatial_dropout1… │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 576,881 (2.20 MB)

...

INFO:tensorflow:Assets written to: /tmp/tmp0x18yxwo/assets


INFO:tensorflow:Assets written to: /tmp/tmp0x18yxwo/assets


Saved artifact at '/tmp/tmp0x18yxwo'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 296, 256, 1), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 129), dtype=tf.float32, name=None)
Captures:
  125400587873616: TensorSpec(shape=(), dtype=tf.resource, name=None)
  125400587874576: TensorSpec(shape=(), dtype=tf.resource, name=None)
  125400587873808: TensorSpec(shape=(), dtype=tf.resource, name=None)
  125400587875344: TensorSpec(shape=(), dtype=tf.resource, name=None)
  125400587874768: TensorSpec(shape=(), dtype=tf.resource, name=None)
  125400587874192: TensorSpec(shape=(), dtype=tf.resource, name=None)
  125400587874384: TensorSpec(shape=(), dtype=tf.resource, name=None)
  125400587874960: TensorSpec(shape=(), dtype=tf.resource, name=None)
  125400587875728: TensorSpec(shape=(), dtype=tf.resource, name=None)
  125400587876688: TensorSpec(shape=(), dtype=tf.resource, name=None)
  125400587876

W0000 00:00:1772220261.350461   91365 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1772220261.350474   91365 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1772220261.350654   91365 reader.cc:83] Reading SavedModel from: /tmp/tmp0x18yxwo
I0000 00:00:1772220261.351466   91365 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1772220261.351470   91365 reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmp0x18yxwo
I0000 00:00:1772220261.360233   91365 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled
I0000 00:00:1772220261.361940   91365 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:1772220261.423508   91365 loader.cc:220] Running initialization op on SavedModel bundle at path: /tmp/tmp0x18yxwo
I0000 00:00:1772220261.438863   91365 loader.cc:471] SavedModel load for tags { serve }; Status: success: OK. Took 88216 microseconds.
I0000 00:00:1772220261.460028   91365

TFLite model saved as guitarmidi.tflite
